# Spark Setup

In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

HOST_IP = "192.168.64.1"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .getOrCreate()
)

print("Debug: SparkSession has been created successfully.")

Debug: SparkSession has been created successfully.


Accepting streams

In [2]:

# Create a JSON schema that matches the payload from producer for easier handling

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])

def read_camera_stream(topic, producer):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{HOST_IP}:9092")
        .option("subscribe", topic)
        .load()
        # The value from Kafka is in bytes, so we can cast it to a string
        .selectExpr("CAST(value AS STRING) as json_value")
        # Parse the string into the columns using the struct schema we defined
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        # Convert timestamp to proper Spark timestamp type
        .withColumn("event_time", to_timestamp(col("timestamp")))
        # Tag each event with its source
        .withColumn("source", lit(producer))
        # Watermarking to handle late data,
        .withWatermark("event_time", "10 minutes")
    )

camera_stream_a = read_camera_stream("camera-events-A", "camera A")
camera_stream_b = read_camera_stream("camera-events-B", "camera B")
camera_stream_c = read_camera_stream("camera-events-C", "camera C")

print("Debug: Kafka streams have been created for all three cameras.")

Debug: Kafka streams have been created for all three cameras.


join them with each other so easy process i guess idk ill figure out why later

In [3]:
camera_df = (
    spark.read.csv(f"{Path('..')}/data/camera.csv", header=True, inferSchema=True)
    .select("camera_id", "position", "speed_limit")
)

camera_df.show()
print(f"Debug: Camera loaded: {camera_df.count()} cameras.")

def join_stream_with_camera(stream):
    return stream.join(camera_df, on="camera_id", how="inner")

joined_stream_a = join_stream_with_camera(camera_stream_a)
joined_stream_b = join_stream_with_camera(camera_stream_b)
joined_stream_c = join_stream_with_camera(camera_stream_c)

print("Debug: Streams have been joined with camera data.")

+---------+--------+-----------+
|camera_id|position|speed_limit|
+---------+--------+-----------+
|        1|   152.5|        110|
|        2|   153.5|        110|
|        3|   154.5|         90|
+---------+--------+-----------+

Debug: Camera loaded: 3 cameras.
Debug: Streams have been joined with camera data.


In [ ]:
def get_instant_violations(stream):
    return (
        stream
        .filter(col("speed_reading") > col("speed_limit"))
        .withColumn("violation_type", lit("instantaneous"))
        .withColumn("violation_date", to_date(col("event_time")))
        .select(
            "car_plate",
            "violation_date",
            "violation_type",
            "camera_id",
            col("speed_reading").alias("speed_recorded"),
            "speed_limit",
            col("event_time").cast("string").alias("event_time"),
            "source" 
        )
    )

camera_a_instant_violations = get_instant_violations(joined_stream_a)
camera_b_instant_violations = get_instant_violations(joined_stream_b)
camera_c_instant_violations = get_instant_violations(joined_stream_c)

# Combine all instant violations into one stream
all_instant_violations = (
    camera_a_instant_violations
    .union(camera_b_instant_violations)
    .union(camera_c_instant_violations)
)

print("Debug: Instantaneous violations have been extracted and combined.")

def log_batch(name):

    def logger(batch_df, batch_id):

        now = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        row_count = batch_df.count()

        print("\n" + "=" * 70)
        print(f"[{now}] {name}")
        print(f"Spark Batch ID: {batch_id}")
        print(f"Rows received: {row_count}")
        print("=" * 70)

        batch_df.show(
            truncate=False
        )

    return logger

instant_query = (
    all_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(log_batch("Instantaneous Violations"))
    .start()
)

spark.streams.awaitAnyTermination()


Debug: Instantaneous violations have been extracted and combined.

[2026-05-11 12:07:23] Instantaneous Violations
Spark Batch ID: 0
Rows received: 0
+---------+--------------+--------------+---------+--------------+-----------+----------+------+
|car_plate|violation_date|violation_type|camera_id|speed_recorded|speed_limit|event_time|source|
+---------+--------------+--------------+---------+--------------+-----------+----------+------+
+---------+--------------+--------------+---------+--------------+-----------+----------+------+


[2026-05-11 12:07:23] Instantaneous Violations
Spark Batch ID: 1
Rows received: 20
+---------+--------------+--------------+---------+--------------+-----------+-------------------+--------+
|car_plate|violation_date|violation_type|camera_id|speed_recorded|speed_limit|event_time         |source  |
+---------+--------------+--------------+---------+--------------+-----------+-------------------+--------+
|QKR 18   |2024-01-01    |instantaneous |1        |127


[2026-05-11 12:07:35] Instantaneous Violations
Spark Batch ID: 7
Rows received: 2
+---------+--------------+--------------+---------+--------------+-----------+--------------------------+--------+
|car_plate|violation_date|violation_type|camera_id|speed_recorded|speed_limit|event_time                |source  |
+---------+--------------+--------------+---------+--------------+-----------+--------------------------+--------+
|EBZ 5064 |2024-01-01    |instantaneous |2        |138.3         |110        |2024-01-01 08:51:52.352273|camera B|
|BCZ 29   |2024-01-01    |instantaneous |2        |158.0         |110        |2024-01-01 08:51:55.011053|camera B|
+---------+--------------+--------------+---------+--------------+-----------+--------------------------+--------+


[2026-05-11 12:07:36] Instantaneous Violations
Spark Batch ID: 8
Rows received: 1
